In [1]:
# === Cell 0: Sanity check ===
from random import randint

print(randint(1, 100))

75


In [1]:
# === Cell 1: Mount Drive ===
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# === Cell 2: Drive write test (disabled) ===
# import os
# test_dir = '/content/drive/MyDrive/lmu_adl_eval_history'
# os.makedirs(test_dir, exist_ok=True)
# test_file = os.path.join(test_dir, '_write_test.txt')
# with open(test_file, 'w') as f:
#     f.write('ok')
# print(f'Write successful: {os.path.exists(test_file)}')
# os.remove(test_file)

# SGHMC & BAOA Evaluation: Pretrained-Prior vs Zero-Centered-Prior

Runs both **automated metrics** (BLEU / ROUGE / Perplexity) and **LLM-as-judge**
(Quality / Diversity / Relevance) on SGHMC and BAOA models with two prior configurations.

## 1. Setup

In [9]:
# === Cell 5: Clone/update repo ===
import os
if not os.path.isdir('/content/adl-bnn-textgen'):
    !git clone https://github.com/ssophiee/adl-bnn-textgen /content/adl-bnn-textgen
else:
    !git -C /content/adl-bnn-textgen pull --ff-only
    print('Repo updated.')

from google.colab import drive
drive.mount('/content/drive')

Already up to date.
Repo updated.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
# === Cell 6: Find project root ===
import os

def find_project_root():
    candidates = [
        os.getcwd(),
        os.path.abspath(os.path.join(os.getcwd(), '..')),
        '/content/adl-bnn-textgen',
    ]
    for c in candidates:
        if os.path.isfile(os.path.join(c, 'config.py')):
            return c
    raise FileNotFoundError('Could not find project root (looked for config.py)')

PROJECT_ROOT = find_project_root()
print(f'Project root: {PROJECT_ROOT}')
!git -C "{PROJECT_ROOT}" log --oneline -3

Project root: /content/adl-bnn-textgen
bf75fda (HEAD -> main, origin/main, origin/HEAD) fix: check all three score keys before aggregating results
8f1b10e fix: reduce Qwen batch_size to 8 to fit KV cache in T4 VRAM
f6c10a6 diag: add GPU memory report after Qwen model load


In [11]:
# === Cell 7: Write .env ===
env_content = f"""BASE_DIR={PROJECT_ROOT}
MODEL_PATH=/content/drive/MyDrive/baseline/baseline_model_2k.pt
META_PATH=/content/drive/MyDrive/baseline/nanogpt_meta.pkl
DATA_DIR={PROJECT_ROOT}/data/tokenized
BNN_MODEL_PATH=/content/drive/MyDrive/baseline/baseline_model_2k.pt
DEVICE=cuda
WANDB_AVAILABLE=false
"""

env_path = os.path.join(PROJECT_ROOT, '.env')
with open(env_path, 'w') as f:
    f.write(env_content)

print(env_content)
print(f'Environment file written to: {env_path}')

BASE_DIR=/content/adl-bnn-textgen
MODEL_PATH=/content/drive/MyDrive/baseline/baseline_model_2k.pt
META_PATH=/content/drive/MyDrive/baseline/nanogpt_meta.pkl
DATA_DIR=/content/adl-bnn-textgen/data/tokenized
BNN_MODEL_PATH=/content/drive/MyDrive/baseline/baseline_model_2k.pt
DEVICE=cuda
WANDB_AVAILABLE=false

Environment file written to: /content/adl-bnn-textgen/.env


In [12]:
# === Cell 8: Install deps + HF login ===
!pip install -q python-dotenv posteriors evaluate rouge_score datasets bitsandbytes


In [13]:
# === Cell 9: GPU memory diagnostic helper ===
import torch, psutil, os

def gpu_mem_report(tag=""):
    """Print GPU and CPU memory usage."""
    prefix = f"[{tag}] " if tag else ""
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        free = total - alloc
        print(f"{prefix}GPU: {alloc:.2f}GB allocated, {reserved:.2f}GB reserved, {free:.2f}GB free / {total:.2f}GB total")
    else:
        print(f"{prefix}No GPU available")
    ram = psutil.virtual_memory()
    print(f"{prefix}RAM: {ram.used/1024**3:.1f}GB used / {ram.total/1024**3:.1f}GB total ({ram.percent}%)")

gpu_mem_report("After setup")

[After setup] No GPU available
[After setup] RAM: 0.8GB used / 12.7GB total (8.7%)


## 2. Discover models

In [14]:
# === Cell 11: Discover models ===
import glob, os

# --- SGHMC models ---
SGHMC_BASE = '/content/drive/MyDrive/samplers/sghmc_sampler'
sghmc_pretrained = sorted(glob.glob(f'{SGHMC_BASE}/pretrained-prior/run_*/sghmc_model.pt'))
sghmc_zero = sorted(glob.glob(f'{SGHMC_BASE}/zero-centered-prior/run_*/sghmc_model.pt'))

print(f'SGHMC pretrained-prior ({len(sghmc_pretrained)}):')
for p in sghmc_pretrained:
    print(f'  {p}')
print(f'SGHMC zero-centered-prior ({len(sghmc_zero)}):')
for p in sghmc_zero:
    print(f'  {p}')

# --- BAOA models ---
BAOA_BASE = '/content/drive/MyDrive/samplers/baoa_sampler'
baoa_pretrained = sorted(glob.glob(f'{BAOA_BASE}/pretrained-prior/run_*/baoa_model.pt'))
baoa_zero = sorted(glob.glob(f'{BAOA_BASE}/zero-centered-prior/run_*/baoa_model.pt'))

print(f'\nBAOA pretrained-prior ({len(baoa_pretrained)}):')
for p in baoa_pretrained:
    print(f'  {p}')
print(f'BAOA zero-centered-prior ({len(baoa_zero)}):')
for p in baoa_zero:
    print(f'  {p}')

# Build combined lists: models + their sampler types
all_models = sghmc_pretrained + sghmc_zero + baoa_pretrained + baoa_zero
all_sampler_types = (
    ['sghmc'] * (len(sghmc_pretrained) + len(sghmc_zero))
    + ['baoa'] * (len(baoa_pretrained) + len(baoa_zero))
)

print(f'\nTotal models: {len(all_models)}')
assert len(all_models) == 8, f'Expected 8 models, found {len(all_models)}. Check the paths above.'

SGHMC pretrained-prior (2):
  /content/drive/MyDrive/samplers/sghmc_sampler/pretrained-prior/run_20260224-230657/sghmc_model.pt
  /content/drive/MyDrive/samplers/sghmc_sampler/pretrained-prior/run_20260224-232547/sghmc_model.pt
SGHMC zero-centered-prior (2):
  /content/drive/MyDrive/samplers/sghmc_sampler/zero-centered-prior/run_20260224-231606/sghmc_model.pt
  /content/drive/MyDrive/samplers/sghmc_sampler/zero-centered-prior/run_20260224-233423/sghmc_model.pt

BAOA pretrained-prior (2):
  /content/drive/MyDrive/samplers/baoa_sampler/pretrained-prior/run_20260207-192540/baoa_model.pt
  /content/drive/MyDrive/samplers/baoa_sampler/pretrained-prior/run_20260225-101215/baoa_model.pt
BAOA zero-centered-prior (2):
  /content/drive/MyDrive/samplers/baoa_sampler/zero-centered-prior/run_20260207-232141/baoa_model.pt
  /content/drive/MyDrive/samplers/baoa_sampler/zero-centered-prior/run_20260225-102111/baoa_model.pt

Total models: 8


## 3. LLM-as-Judge Evaluation (Generation + Scoring)

In [27]:
# === Cell 13: Import pipeline + extract prompts ===
import sys, json, importlib
sys.path.insert(0, PROJECT_ROOT)

# Load .env explicitly (cwd is /content, but .env is in PROJECT_ROOT)
from dotenv import load_dotenv
load_dotenv(os.path.join(PROJECT_ROOT, '.env'), override=True)

# Extract prompts from the testing results file
gen_results_path = os.path.join(PROJECT_ROOT, 'results/evaluation/llm_judge/external_data/generation_results_testing.json')
with open(gen_results_path, 'r') as f:
    gen_data = json.load(f)

test_prompts = sorted(set(v['prompt'] for v in gen_data.values() if 'prompt' in v))
print(f'Extracted {len(test_prompts)} unique prompts')
for i, p in enumerate(test_prompts):
    print(f'  {i+1}. {p[:60]}...')

# Save prompts for reference
prompts_out = os.path.join(PROJECT_ROOT, 'results/evaluation/llm_judge/external_prompts.json')
with open(prompts_out, 'w') as f:
    json.dump(test_prompts, f, indent=2)

# Force reload ALL relevant modules to pick up git-pulled changes
for mod_name in list(sys.modules.keys()):
    if mod_name.startswith('scripts'):
        del sys.modules[mod_name]

from scripts.llm_evaluation import run_evaluation_pipeline

# Verify the 4-bit code is present in the loaded module
import scripts.llm_evaluation as _mod
import inspect
src = inspect.getsource(_mod._evaluate_with_local_qwen)
if 'BitsAndBytesConfig' in src and 'load_in_4bit' in src:
    print('\n4-bit quantization code CONFIRMED in loaded module')
else:
    print('\nWARNING: 4-bit code NOT found! Module may be stale.')
    print('First 200 chars of _evaluate_with_local_qwen:')
    print(src[:200])

# Release any leftover GPU memory from previous runs
torch.cuda.empty_cache()
import gc; gc.collect()
gpu_mem_report("Before LLM judge (after cache clear)")

Extracted 15 unique prompts
  1. And so it was, that in those days,...
  2. HAMLET: To be, or not to be, that is the question:...
  3. In such a manner, one might say,...
  4. In the beginning, there was darkness and light,...
  5. JULIET: O Romeo, Romeo! Wherefore art thou Romeo?...
  6. Love conquers all, they say, but I have found...
  7. MACBETH: Is this a dagger which I see before me,...
  8. OPHELIA: My lord, I have remembrances of yours,...
  9. Once upon a time in a kingdom far away,...
  10. ROMEO: But soft! What light through yonder window breaks?...
  11. She looked into his eyes and saw...
  12. The greatest treasure in all the world is not...
  13. The king turned to his advisor and...
  14. The old king looked upon his realm and said:...
  15. The thing that happened next was most...

4-bit quantization code CONFIRMED in loaded module
[Before LLM judge (after cache clear)] GPU: 0.01GB allocated, 6.70GB reserved, 14.55GB free / 14.56GB total
[Before LLM judge (after cache 

In [28]:
# === Cell 14: Run LLM judge pipeline ===
model_types = ['bayesian'] * len(all_models)

DRIVE_OUTPUT = '/content/drive/MyDrive/lmu_adl_eval_history'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

# Override default grid to match previous evaluation
from scripts.llm_evaluation import EvaluationConfig
EvaluationConfig.DEFAULT_TEMPERATURE = [0.3, 0.8]
EvaluationConfig.DEFAULT_TOP_K = [10, 50]
EvaluationConfig.DEFAULT_NUM_SAMPLES = [10, 30]

# Use first 4 prompts only
prompts_subset = test_prompts[:4]
print(f'Using {len(prompts_subset)} prompts:')
for i, p in enumerate(prompts_subset):
    print(f'  {i+1}. {p[:60]}...')

aggregated, full_results = run_evaluation_pipeline(
    test_prompts=prompts_subset,
    model_paths=all_models,
    model_types=model_types,
    change_params=True,
    output_path=f'{DRIVE_OUTPUT}/generation_results.json',
    use_local_qwen=True,
    device='cuda',
    use_external_data=True,
)

Using 4 prompts:
  1. And so it was, that in those days,...
  2. HAMLET: To be, or not to be, that is the question:...
  3. In such a manner, one might say,...
  4. In the beginning, there was darkness and light,...

################################################################################
# LLM Evaluation Pipeline
# Started at: 2026-03-04 17:21:24
################################################################################

Using 4 provided prompts

Loaded 256 previous results from: /content/drive/MyDrive/lmu_adl_eval_history/generation_results.json

Starting Text Generation Phase
Total generations to perform: 256
Already completed (resuming): 256
Remaining: 0
Models: 8
Prompts: 4
Parameter combinations: 8


--- Processing model 1/8: sghmc_model.pt (type: bayesian) ---
  All 32 generations cached — skipping model load

--- Processing model 2/8: sghmc_model.pt (type: bayesian) ---
  All 32 generations cached — skipping model load

--- Processing model 3/8: sghmc_model.pt (ty

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Model loaded successfully! (GPU: 5.5GB, 4-bit quantized: True)

Scoring 36 texts in 6 batches (batch_size=8)

Evaluating prompt 1/2: 'HAMLET: To be, or not to be, that is the question:...'
  batch 1/6: scored 8/8 texts (74.5s/batch, ETA: 6.2min)
  batch 2/6: scored 8/8 texts (72.9s/batch, ETA: 4.9min)
  batch 3/6: scored 8/8 texts (73.0s/batch, ETA: 3.7min)
  batch 4/6: scored 8/8 texts (72.3s/batch, ETA: 2.4min)
  batch 5/6: scored 3/3 texts (62.5s/batch, ETA: 1.0min)
Evaluating prompt 2/2: 'And so it was, that in those days,...'
  batch 6/6: scored 1/1 texts (53.5s/batch, ETA: 0.0min)

Evaluation Phase Complete: 36 texts evaluated


Aggregating Results


--- Summary Statistics ---

Model: sghmc_model.pt
  Overall Quality: 4.19
  Overall Diversity: 4.81
  Overall Relevance: 5.33
  Total Generations: 32

Model: sghmc_model.pt
  Overall Quality: 4.05
  Overall Diversity: 5.11
  Overall Relevance: 6.20
  Total Generations: 32

Model: sghmc_model.pt
  Overall Quality: 3.83
  Overall Diver

In [29]:
# === Cell 15: LLM judge summary ===
from pathlib import Path

print('\n' + '='*80)
print('LLM-Judge Results Summary')
print('='*80)
for model_path, scores in aggregated.items():
    run_dir = Path(model_path).parent.name
    prior_type = Path(model_path).parent.parent.name
    print(f'\n{prior_type} / {run_dir}:')
    print(f'  Quality:   {scores["overall_avg_quality"]:.2f}')
    print(f'  Diversity: {scores["overall_avg_diversity"]:.2f}')
    print(f'  Relevance: {scores["overall_avg_relevance"]:.2f}')
    print(f'  Generations: {scores["total_generations"]}')

gpu_mem_report("After LLM judge")


LLM-Judge Results Summary

pretrained-prior / run_20260224-230657:
  Quality:   4.19
  Diversity: 4.81
  Relevance: 5.33
  Generations: 32

pretrained-prior / run_20260224-232547:
  Quality:   4.05
  Diversity: 5.11
  Relevance: 6.20
  Generations: 32

zero-centered-prior / run_20260224-231606:
  Quality:   3.83
  Diversity: 4.89
  Relevance: 5.81
  Generations: 32

zero-centered-prior / run_20260224-233423:
  Quality:   4.61
  Diversity: 5.66
  Relevance: 6.30
  Generations: 32

pretrained-prior / run_20260207-192540:
  Quality:   4.33
  Diversity: 5.27
  Relevance: 6.25
  Generations: 32

pretrained-prior / run_20260225-101215:
  Quality:   4.14
  Diversity: 5.19
  Relevance: 5.83
  Generations: 32

zero-centered-prior / run_20260207-232141:
  Quality:   4.12
  Diversity: 4.95
  Relevance: 5.83
  Generations: 32

zero-centered-prior / run_20260225-102111:
  Quality:   3.91
  Diversity: 5.05
  Relevance: 5.61
  Generations: 32
[After LLM judge] GPU: 3.43GB allocated, 8.65GB reserved,

## 4. Automated Metrics (BLEU / ROUGE / Perplexity)

In [17]:
# === Cell 17: Load tokenizer for automated metrics ===
import sys, torch, gc
from pathlib import Path
from dotenv import load_dotenv

sys.path.insert(0, PROJECT_ROOT)
load_dotenv(os.path.join(PROJECT_ROOT, '.env'), override=True)

# Free Qwen model from GPU before loading Bayesian models + GPT-2
torch.cuda.empty_cache()
gc.collect()

from src.nanogpt_utils import load_model, load_tokenizer
from src.generation_utils import load_checkpoint_for_generation
from scripts.bayesian_evaluator import BayesianNanoGPTEvaluator, evaluate_bayesian_splits

META_PATH = '/content/drive/MyDrive/baseline/nanogpt_meta.pkl'
DATA_DIR  = os.path.join(PROJECT_ROOT, 'data/tokenized')
DEVICE    = 'cpu'  # NanoGPT is tiny (10.65M params), CPU is fine and avoids GPU memory issues

stoi, itos = load_tokenizer(Path(META_PATH))
print(f'Vocab size: {len(itos)}, Device: {DEVICE}')

gpu_mem_report("Before automated metrics")

Vocab size: 65, Device: cpu
[Before automated metrics] No GPU available
[Before automated metrics] RAM: 1.0GB used / 12.7GB total (10.3%)


In [ ]:
print(1)

In [21]:
# === Cell 18: Run automated metrics (BLEU/ROUGE/Perplexity) ===
import json, time, gc
import numpy as np

DRIVE_OUTPUT = '/content/drive/MyDrive/lmu_adl_eval_history'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
AUTO_METRICS_PATH = f'{DRIVE_OUTPUT}/automated_metrics_v2.json'

def _save_auto_checkpoint():
    with open(AUTO_METRICS_PATH, 'w') as f:
        json.dump(auto_results, f, indent=2, default=str)

# Config
NUM_TEXT_SAMPLES = 4
PROMPT_LENGTH = 30
GENERATION_LENGTH = 30
PPL_BATCH_SIZE = 16
PPL_MAX_BATCHES = 4  # small for CPU speed

print(f'Config: {NUM_TEXT_SAMPLES} random windows/split, '
      f'prompt={PROMPT_LENGTH}, gen={GENERATION_LENGTH}, '
      f'ppl_batches={PPL_MAX_BATCHES} (no GPT-2 external)')

# Auto-resume from Drive
auto_results = {}
if os.path.exists(AUTO_METRICS_PATH):
    with open(AUTO_METRICS_PATH, 'r') as f:
        auto_results = json.load(f)
    print(f'Resumed {len(auto_results)} previous model results from Drive')

auto_start_time = time.time()
total_models = len(all_models)
models_done = 0

for model_idx, (model_path, sampler_type) in enumerate(zip(all_models, all_sampler_types)):
    run_dir = Path(model_path).parent.name
    prior_type = Path(model_path).parent.parent.name
    label = f'{sampler_type}/{prior_type}/{run_dir}'

    existing = auto_results.get(label, {})
    all_done = all(
        isinstance(existing.get(s), dict) and 'bleu' in existing.get(s, {})
        for s in ['train', 'val']
    )
    if all_done:
        print(f'\nSKIP (cached): {label}')
        models_done += 1
        continue

    print(f'\n{"="*60}')
    print(f'[{model_idx+1}/{total_models}] {label}')
    print(f'{"="*60}')

    model_load_t0 = time.time()
    model, _ = load_model(Path('/content/drive/MyDrive/baseline/baseline_model_2k.pt'), DEVICE)
    ckpt = load_checkpoint_for_generation(model_path, device=DEVICE)
    collected_samples = ckpt.get('collected_samples', [])
    num_posterior = min(10, len(collected_samples))
    print(f'  Loaded in {time.time()-model_load_t0:.1f}s | posterior samples: {num_posterior}')

    evaluator = BayesianNanoGPTEvaluator(
        model=model, stoi=stoi, itos=itos,
        sampler_type=sampler_type,
        collected_samples=collected_samples,
        device=DEVICE,
        num_posterior_samples=num_posterior,
    )

    if label not in auto_results:
        auto_results[label] = {'model_path': model_path, 'prior_type': prior_type, 'sampler_type': sampler_type}

    for split in ['train', 'val']:
        if isinstance(auto_results[label].get(split), dict) and 'bleu' in auto_results[label].get(split, {}):
            print(f'  [{split}] cached, skipping')
            continue

        split_t0 = time.time()
        data = evaluator.load_data(DATA_DIR, split)
        print(f'  [{split}] data: {len(data)} tokens')

        # 1) Internal perplexity (BMA)
        print(f'  [{split}] internal perplexity ({PPL_MAX_BATCHES} batches)...', end=' ', flush=True)
        t0 = time.time()
        ppl = evaluator.calculate_perplexity_internal(data, PPL_BATCH_SIZE, max_batches=PPL_MAX_BATCHES)
        print(f'{ppl:.1f} ({time.time()-t0:.1f}s)')

        # 2) Generate samples + BLEU/ROUGE
        print(f'  [{split}] generating {NUM_TEXT_SAMPLES} BMA samples...', end=' ', flush=True)
        t0 = time.time()
        refs, preds = evaluator.generate_samples_for_metrics(data, NUM_TEXT_SAMPLES, PROMPT_LENGTH, GENERATION_LENGTH)
        print(f'{len(refs)} pairs ({time.time()-t0:.1f}s)')
        for i in range(min(2, len(refs))):
            print(f'           ref:  "{refs[i][:50]}"')
            print(f'           pred: "{preds[i][:50]}"')

        split_result = {
            'split': split,
            'sampler_type': sampler_type,
            'total_tokens': len(data),
            'num_posterior_samples': num_posterior,
            'perplexity': ppl if ppl is not None else 0.0,
        }

        if refs and preds:
            bleu = evaluator.calculate_bleu_score(refs, preds)
            split_result['bleu'] = bleu if bleu is not None else 0.0
            rouge = evaluator.calculate_rouge_score(refs, preds)
            split_result.update(rouge)
            print(f'  [{split}] BLEU={split_result["bleu"]:.4f}  ROUGE-2={split_result.get("rouge2",0):.4f}')
        else:
            split_result.update({'bleu': 0.0, 'rouge1': 0.0, 'rouge2': 0.0, 'rougeL': 0.0})
            print(f'  [{split}] no valid generations')

        split_result['elapsed_seconds'] = round(time.time() - split_t0, 1)
        auto_results[label][split] = split_result
        _save_auto_checkpoint()
        print(f'  [{split}] done in {split_result["elapsed_seconds"]}s (saved)')

    models_done += 1
    elapsed = time.time() - auto_start_time
    eta_min = elapsed / models_done * (total_models - models_done) / 60
    print(f'  [{models_done}/{total_models} done, elapsed: {elapsed/60:.1f}min, ETA: {eta_min:.1f}min]')

    del model, ckpt, collected_samples, evaluator
    gc.collect()

total_time = time.time() - auto_start_time
print(f'\nAll {len(auto_results)} models evaluated in {total_time/60:.1f}min.')

Config: 4 random windows/split, prompt=30, gen=30, ppl_batches=4 (no GPT-2 external)

[1/8] sghmc/pretrained-prior/run_20260224-230657
Model arguments: {'n_layer': 6, 'n_head': 6, 'n_embd': 384, 'block_size': 256, 'bias': False, 'vocab_size': 65, 'dropout': 0.2}
number of parameters: 10.65M
Loaded 100 collected samples from checkpoint
  Loaded in 25.9s | posterior samples: 10
  [train] data: 1003854 tokens
  [train] internal perplexity (4 batches)... 3.3 (80.0s)
  [train] generating 4 BMA samples... 4 pairs (37.2s)
           ref:  "ar services
Past and to come,"
           pred: "parted down to venture
A whom"
           ref:  "r, that for the poorest piece"
           pred: "will but the airy of honesty."
  [train] BLEU=0.1571  ROUGE-2=0.1850
  [train] done in 117.5s (saved)
  [val] data: 111540 tokens
  [val] internal perplexity (4 batches)... 4.3 (78.5s)
  [val] generating 4 BMA samples... 4 pairs (36.3s)
           ref:  ":
Proceed.

Tailor:

GRUMIO:
I"
           pred: ":
Speak n

## 5. Save Results

In [22]:
# === Cell 20: Save results ===
from datetime import datetime

print(f'Automated metrics already saved to: {AUTO_METRICS_PATH}')
print(f'LLM-judge results saved to: {DRIVE_OUTPUT}/generation_results*.json')
print(f'\nAll results on Drive at {datetime.now():%Y-%m-%d %H:%M:%S}')

Automated metrics already saved to: /content/drive/MyDrive/lmu_adl_eval_history/automated_metrics_v2.json
LLM-judge results saved to: /content/drive/MyDrive/lmu_adl_eval_history/generation_results*.json

All results on Drive at 2026-03-04 19:09:58


## 6. Quick Comparison Table

In [23]:
# === Cell 22: Comparison table ===
import pandas as pd

rows = []
for label, data in auto_results.items():
    for split in ['train', 'val']:
        m = data.get(split, {})
        if isinstance(m, dict) and 'error' not in m:
            rows.append({
                'model': label,
                'split': split,
                'bleu': m.get('bleu', 0),
                'rouge1': m.get('rouge1', 0),
                'rouge2': m.get('rouge2', 0),
                'rougeL': m.get('rougeL', 0),
                'ppl_internal': m.get('perplexity', 0),
                'ppl_gpt2': m.get('perplexity_external_gpt2', 0),
            })

df = pd.DataFrame(rows)
if not df.empty:
    display(df.round(4))
else:
    print('No results to display.')

,model,split,bleu,rouge1,rouge2,rougeL,ppl_internal,ppl_gpt2
0,sghmc/pretrained-prior/run_20260224-230657,train,0.1571,0.5859,0.1850,0.3483,3.2811,0
1,sghmc/pretrained-prior/run_20260224-230657,val,0.0680,0.4310,0.1482,0.3039,4.3297,0
2,sghmc/pretrained-prior/run_20260224-232547,train,0.1843,0.5474,0.1863,0.3681,3.3576,0
3,sghmc/pretrained-prior/run_20260224-232547,val,0.0000,0.4538,0.0887,0.2824,4.3901,0
4,sghmc/zero-centered-prior/run_20260224-231606,train,0.1110,0.5966,0.1909,0.3948,3.2766,0
5,sghmc/zero-centered-prior/run_20260224-231606,val,0.0000,0.5574,0.1667,0.3131,4.3320,0
6,sghmc/zero-centered-prior/run_20260224-233423,train,0.0000,0.5579,0.1572,0.3720,3.4222,0
7,sghmc/zero-centered-prior/run_20260224-233423,val,0.1382,0.5515,0.2315,0.3614,4.4653,0
8,baoa/pretrained-prior/run_20260207-192540,train,0.1845,0.4891,0.2538,0.3547,3.2609,0
9,baoa/pretrained-prior/run_20260207-192540,val,0.0891,0.4917,0.1552,0.3333,4.3103,0
